In [7]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as f
from torch.utils.data import IterableDataset,DataLoader,get_worker_info
import torch.optim as optim
import numpy as np
import zstandard as zstd
import random

if torch.cuda.is_available():
	print("PyTorch is using the GPU")
	GPUCount = torch.cuda.device_count()
	print(f"Found {GPUCount} GPUs")

	for i in range(GPUCount):
		print(f"GPU {i} found: {torch.cuda.get_device_name(i)}")

	device = torch.device("cuda:0")
else:
	print("PyTorch is using the CPU")
	device = torch.device("cpu")

print(f"Selected Device: {device}")

PyTorch is using the GPU
Found 1 GPUs
GPU 0 found: NVIDIA GeForce RTX 5070 Laptop GPU
Selected Device: cuda:0


In [8]:
class SEBlock(nn.Module):
	def __init__(self,channels,reduction=16):
		super().__init__()
		self.squeeze = nn.AdaptiveAvgPool2d(1)
		self.excite = nn.Sequential(
			nn.Linear(channels,channels//reduction,bias=False),
			nn.SiLU(inplace=True),
			nn.Linear(channels//reduction,channels,bias=False),
			nn.Sigmoid()
		)

	def forward(self,x):
		b,c,_,_ = x.size()
		y = self.squeeze(x).view(b,c)
		y = self.excite(y).view(b,c,1,1)
		return x * y.expand_as(x)

class ResidualBlock(nn.Module):
	def __init__(self,NumChannels):
		super().__init__()
		self.conv1 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn1 = nn.BatchNorm2d(NumChannels)

		self.conv2 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn2 = nn.BatchNorm2d(NumChannels)

		self.se = SEBlock(NumChannels)

	def forward(self,x):
		residual = x
		
		x = f.silu(self.bn1(self.conv1(x)))

		x = self.bn2(self.conv2(x))

		x = self.se(x)

		x += residual

		return f.silu(x)
	
class ChessNet(nn.Module):
	def __init__(self):
		super().__init__()

		self.ConvInput = nn.Conv2d(in_channels=16,out_channels=256,kernel_size=3,padding=1)
		self.BnInput = nn.BatchNorm2d(256)

		self.ResTower = nn.Sequential(*[ResidualBlock(256) for _ in range(10)])

		self.ConvValue = nn.Conv2d(in_channels=256,out_channels=32,kernel_size=1)
		self.BnValue = nn.BatchNorm2d(32)

		self.flat = nn.Flatten()

		self.fc1 = nn.Linear(32*8*8,256)
		self.fc2 = nn.Linear(256,1)
		nn.init.normal_(self.fc2.weight,mean=0.0,std=0.01)

		self.PST = nn.Parameter(torch.randn(1,16,8,8)*0.01)

	def forward(self,x):
		RawBoard = x
		x = f.silu(self.BnInput(self.ConvInput(x)))

		x = self.ResTower(x)

		x = f.silu(self.BnValue(self.ConvValue(x)))
		x = self.flat(x)
		x = f.silu(self.fc1(x))

		ComplexEval = self.fc2(x)

		SpatialEval = torch.sum(RawBoard*self.PST,dim=(1,2,3)).view(-1,1)

		return torch.tanh(ComplexEval+SpatialEval)
	
model = ChessNet()
model.to(device)
print(model)

criterion = nn.MSELoss()
optimiser = optim.AdamW(model.parameters(),lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser,mode='min',factor=0.5,patience=10)

ChessNet(
  (ConvInput): Conv2d(16, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (BnInput): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (ResTower): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (se): SEBlock(
        (squeeze): AdaptiveAvgPool2d(output_size=1)
        (excite): Sequential(
          (0): Linear(in_features=256, out_features=16, bias=False)
          (1): SiLU(inplace=True)
          (2): Linear(in_features=16, out_features=256, bias=False)
          (3): Sigmoid()
        )
      )
    )
    (1): ResidualBlock(
      (conv1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1

In [9]:
class ChessDataset(IterableDataset):
	def __init__(self,DataFile,TotalBoards,split='train',ValRatio = 0.1,BufferSize=50000,BoardsSeen=0):
		self.dctx = zstd.ZstdDecompressor()
		self.DataFile = DataFile
		self.TotalBoards = TotalBoards
		self.split = split
		self.ValRatio = ValRatio
		self.BufferSize = BufferSize
		self.SkipBoards = BoardsSeen

	def __len__(self):
		if self.split == 'val':
			return int(self.TotalBoards * self.ValRatio)
		else:
			return self.TotalBoards - int(self.TotalBoards * self.ValRatio)
	
	def __iter__(self):
		WorkerInfo = get_worker_info()
		if WorkerInfo is None:
			WorkerId = 0
			NumWorkers = 1
		else:
			WorkerId = WorkerInfo.id
			NumWorkers = WorkerInfo.num_workers

		GlobalCounter = self.SkipBoards
		SplitCounter = 0
		ValModulo = int(1.0/self.ValRatio)

		Buffer = []
		
		with open(self.DataFile,'rb') as DataFile:
			with self.dctx.stream_reader(DataFile) as stream:

				if self.SkipBoards > 0:
					BytesToSkip = self.SkipBoards * 4100
					
					while BytesToSkip > 0:
						ChunkSize = min(BytesToSkip,4100*10000)
						stream.read(ChunkSize)
						BytesToSkip -= ChunkSize

				while True:
					Chunk = stream.read(4100)

					if not Chunk or len(Chunk) < 4100:
						break

					IsValBoard = (GlobalCounter % ValModulo == 0)
					GlobalCounter += 1

					if self.split == 'train' and IsValBoard:
						continue
					if self.split == 'val' and not IsValBoard:
						continue

					IsMyTurn = (SplitCounter % NumWorkers == WorkerId)
					SplitCounter += 1

					if not IsMyTurn:
						continue

					BoardBytes = Chunk[:4096]
					TargetBytes = Chunk[4096:]

					BoardTensor = np.frombuffer(BoardBytes,dtype=np.float32).reshape(16,8,8)
					TargetTensor = np.frombuffer(TargetBytes,dtype=np.float32)

					BoardTensor = torch.from_numpy(BoardTensor.copy())
					TargetTensor = torch.from_numpy(TargetTensor.copy())

					if self.split == 'train':
						if len(Buffer) < self.BufferSize:
							Buffer.append((BoardTensor,TargetTensor))
						else:
							idx = random.randint(0,self.BufferSize-1)
							YieldItem = Buffer[idx]
							Buffer[idx] = (BoardTensor,TargetTensor)
							yield YieldItem
					else:
						yield BoardTensor,TargetTensor

		if self.split == 'train':
			random.shuffle(Buffer)
			for item in Buffer:
				yield item

In [10]:
if os.path.exists('ChessModel.pth'):
	Checkpoint = torch.load('ChessModel.pth')

	model.load_state_dict(Checkpoint['ModelState'])
	optimiser.load_state_dict(Checkpoint['OptimiserState'])
	BoardsSeen = Checkpoint['BoardsSeen']
	BestLoss = Checkpoint['BestLoss']
else:
	BoardsSeen = 0
	BestLoss = float('inf')

In [11]:
TrainData = ChessDataset('ChessData.zst',TotalBoards=31470592,split='train',BufferSize=50000,BoardsSeen=BoardsSeen)
ValData = ChessDataset('ChessData.zst',TotalBoards=31470592,split='val',BoardsSeen=BoardsSeen)

TrainLoader = DataLoader(TrainData,batch_size=1024,shuffle=False,num_workers=4,pin_memory=True)
ValLoader = DataLoader(ValData,batch_size=1024,shuffle=False,num_workers=2,pin_memory=True)

In [12]:
print('Starting Stream...')

RunningLoss = 0.0
LogInterval = 1000
MaxPatience = 30
PatienceCounter = 0

ValIter = iter(ValLoader)

model.train()
for BatchIdx,(inputs,targets) in enumerate(TrainLoader):
	inputs = inputs.to(device)
	targets = targets.to(device).float().view(-1,1)

	predictions = model(inputs)

	loss = criterion(predictions,targets)
	loss = loss/4
	loss.backward()

	if (BatchIdx + 1) % 4 == 0:
		optimiser.step()
		optimiser.zero_grad()

	RunningLoss += (loss.item()*4)
	BoardsSeen += int(1024*0.9)

	if (BatchIdx+1) % (LogInterval/10) == 0:
		print(f'Processed {(BatchIdx+1)*1024} Boards...')

	if (BatchIdx+1) % LogInterval == 0:
		model.eval()
		ValLossTotal = 0.0
		with torch.no_grad():
			for _ in range(100):
				try:
					ValBoards,ValTargets = next(ValIter)
				except StopIteration:
					ValIter = iter(ValLoader)
					ValBoards,ValTargets = next(ValIter)

				ValBoards = ValBoards.to(device)
				ValTargets = ValTargets.to(device).float().view(-1,1)

				ValPrediction = model(ValBoards)
				ValLoss = criterion(ValPrediction,ValTargets)
				ValLossTotal += ValLoss.item()
				
		AvgLoss = ValLossTotal/100

		scheduler.step(AvgLoss)

		print(f"Batch {BatchIdx+1} | Training Loss: {(RunningLoss/LogInterval):.6f} | Validation Loss: {AvgLoss:.6f}")

		if AvgLoss < BestLoss:
			print(f"Validation Loss has decreased! Saving model!")
			BestLoss = AvgLoss

			Checkpoint = {
				'ModelState':model.state_dict(),
				'OptimiserState':optimiser.state_dict(),
				'BoardsSeen':BoardsSeen,
				'BestLoss':BestLoss
			}

			torch.save(Checkpoint,'ChessModel.pth')
			PatienceCounter = 0
		else:
			PatienceCounter += 1
			print(f'No improvement... Patience: {PatienceCounter}/{MaxPatience}')

			if (PatienceCounter >= MaxPatience):
				print('Patience limit reached. Stopping...')
				break

		RunningLoss = 0.0
		model.train()

Starting Stream...
Processed 102400 Boards...
Processed 204800 Boards...
Processed 307200 Boards...
Processed 409600 Boards...
Processed 512000 Boards...
Processed 614400 Boards...
Processed 716800 Boards...
Processed 819200 Boards...
Processed 921600 Boards...
Processed 1024000 Boards...
Batch 1000 | Training Loss: 0.197105 | Validation Loss: 0.208147
Validation Loss has decreased! Saving model!
Processed 1126400 Boards...
Processed 1228800 Boards...
Processed 1331200 Boards...
Processed 1433600 Boards...
Processed 1536000 Boards...
Processed 1638400 Boards...
Processed 1740800 Boards...
Processed 1843200 Boards...
Processed 1945600 Boards...
Processed 2048000 Boards...
Batch 2000 | Training Loss: 0.191704 | Validation Loss: 0.388501
No improvement... Patience: 1/30
Processed 2150400 Boards...
Processed 2252800 Boards...
Processed 2355200 Boards...
Processed 2457600 Boards...
Processed 2560000 Boards...
Processed 2662400 Boards...
Processed 2764800 Boards...
Processed 2867200 Boards..

KeyboardInterrupt: 

Stopped training process as it took too long. Might continue later.